# 03. t+1 랜덤 진입가격 민감도

기본 `t+1 open` 진입 라벨을 유지하면서, 주문 지연으로 t+1분봉 중간에 진입하는 경우를 보기 위해 `t+1 low~high` 사이 가격을 여러 번 뽑는다. 랜덤 가격은 **라벨 생성용**이며 모델 입력에는 사용하지 않는다.

## 해석상 제한

- t+1 low와 high는 t+1봉 종료 후에만 확정되므로 실시간 feature로 사용할 수 없다.
- OHLC에는 low와 high의 발생 순서와 각 가격의 체류 빈도가 없다. 따라서 균등분포는 실제 체결분포가 아니라 민감도 가정이다.
- 랜덤 진입 이후의 고점만 구분할 수 없으므로 이 실험은 t+1 전체 high를 사용한다. 진입보다 먼저 발생한 high를 TP로 셀 수 있어 결과가 낙관적일 수 있다.
- 단일 랜덤값보다 여러 번 추출한 평균 TP 확률을 soft label로 사용하는 편이 재현성과 정보량이 높다.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'AGENT.md').is_file() and (candidate / 'README.md').is_file():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')

PROJECT_ROOT = find_project_root()
OUTPUT_ROOT = (PROJECT_ROOT / '../../results/preprocessing').resolve()
BASE_VERSION = 'ohlc_60m_tp3pct_v1'
VERSION = 'ohlc_60m_tp3pct_random_entry_v1'
HORIZONS = [1, 3, 5]
TAKE_PROFIT_RETURN = 0.03
RANDOM_SEED = 42
RANDOM_DRAWS = 64

metadata_path = OUTPUT_ROOT / f'{BASE_VERSION}_metadata.parquet'
metadata = pd.read_parquet(metadata_path)
print('metadata:', metadata.shape)
print('seed/draws:', RANDOM_SEED, RANDOM_DRAWS)

metadata: (64144, 20)
seed/draws: 42 64


## 1. t+1 범위에서 진입가격 추출

1분 MFE·MAE는 `t+1 high/low`를 `t+1 open`으로 나눈 값이므로 원래 가격 범위를 정확히 복원할 수 있다. 고정 seed로 각 샘플당 64개 균등 가격을 추출한다.

In [2]:
base_entry = metadata['entry_open'].to_numpy(dtype='float64')
entry_low = base_entry * (1.0 + metadata['mae_1m'].to_numpy(dtype='float64'))
entry_high = base_entry * (1.0 + metadata['mfe_1m'].to_numpy(dtype='float64'))
assert np.less_equal(entry_low, entry_high).all()
assert np.less_equal(entry_low, base_entry + 1e-9).all()
assert np.greater_equal(entry_high, base_entry - 1e-9).all()

rng = np.random.default_rng(RANDOM_SEED)
uniform_draws = rng.random((len(metadata), RANDOM_DRAWS), dtype=np.float32)
random_entries = entry_low[:, None] + uniform_draws * (entry_high - entry_low)[:, None]

range_return = entry_high / entry_low - 1.0
print(f't+1 low~high 범위 중앙값: {np.median(range_return):.3%}')
print(f't+1 low~high 범위 90% 분위수: {np.quantile(range_return, 0.90):.3%}')

t+1 low~high 범위 중앙값: 0.926%
t+1 low~high 범위 90% 분위수: 3.252%


## 2. Open 라벨과 랜덤 진입 라벨 비교

첫 번째 추출값은 재현 가능한 hard-label 비교용으로만 사용하고, 64회 평균을 학습 후보 soft label로 저장한다.

In [3]:
variant = metadata[['sample_id', 'session', 'symbol', 'decision_timestamp', 'entry_timestamp', 'entry_open']].copy()
variant['entry_low_t1'] = entry_low.astype('float32')
variant['entry_high_t1'] = entry_high.astype('float32')
variant['random_entry_seed42'] = random_entries[:, 0].astype('float32')
variant['uniform_entry_mean'] = random_entries.mean(axis=1).astype('float32')

comparison_rows = []
for horizon in HORIZONS:
    future_max_high = base_entry * (1.0 + metadata[f'mfe_{horizon}m'].to_numpy(dtype='float64'))
    hit_matrix = future_max_high[:, None] >= random_entries * (1.0 + TAKE_PROFIT_RETURN)
    soft_probability = hit_matrix.mean(axis=1)
    random_hard = hit_matrix[:, 0]
    base_hard = metadata[f'target_tp3_{horizon}m'].to_numpy(dtype=bool)
    variant[f'p_tp3_random_entry_{horizon}m'] = soft_probability.astype('float32')
    variant[f'target_tp3_random_entry_{horizon}m_seed42'] = random_hard.astype('int8')
    comparison_rows.append({
        'horizon': f'{horizon}m',
        'open_positive_rate': base_hard.mean(),
        'one_random_positive_rate': random_hard.mean(),
        'mean_soft_probability': soft_probability.mean(),
        'soft_probability_ge_50pct': (soft_probability >= 0.5).mean(),
        'hard_label_agreement': (base_hard == random_hard).mean(),
        'open_positive_to_random_negative': (base_hard & ~random_hard).mean(),
        'open_negative_to_random_positive': (~base_hard & random_hard).mean(),
    })

comparison = pd.DataFrame(comparison_rows).set_index('horizon')
display(comparison.style.format('{:.3%}'))

,open_positive_rate,one_random_positive_rate,mean_soft_probability,soft_probability_ge_50pct,hard_label_agreement,open_positive_to_random_negative,open_negative_to_random_positive
horizon,,,,,,,
1m,4.326%,3.498%,3.514%,2.289%,95.325%,2.752%,1.924%
3m,11.878%,11.169%,11.136%,10.241%,93.572%,3.569%,2.859%
5m,17.264%,16.744%,16.746%,16.078%,93.006%,3.757%,3.236%


In [4]:
session_comparison = variant.groupby('session')[[f'p_tp3_random_entry_{horizon}m' for horizon in HORIZONS]].mean()
display(session_comparison.style.format('{:.3%}'))
display(variant.head())

,p_tp3_random_entry_1m,p_tp3_random_entry_3m,p_tp3_random_entry_5m
session,,,
session_2026-07-07,1.523%,6.407%,10.222%
session_2026-07-08,2.945%,11.087%,17.441%
session_2026-07-09,2.806%,9.322%,14.462%
session_2026-07-10,5.883%,15.914%,22.721%
session_2026-07-13,4.222%,12.092%,17.786%
session_2026-07-14,3.408%,10.719%,15.650%
session_2026-07-15,4.247%,12.851%,19.098%
session_2026-07-16,2.794%,10.444%,16.096%
session_2026-07-17,1.896%,6.950%,11.039%


,sample_id,session,symbol,decision_timestamp,entry_timestamp,entry_open,entry_low_t1,entry_high_t1,random_entry_seed42,uniform_entry_mean,p_tp3_random_entry_1m,target_tp3_random_entry_1m_seed42,p_tp3_random_entry_3m,target_tp3_random_entry_3m_seed42,p_tp3_random_entry_5m,target_tp3_random_entry_5m_seed42
0,0,session_2026-07-07,ALM,2026-07-07 14:24:00+00:00,2026-07-07 14:25:00+00:00,14.920,14.860,14.920,14.865355,14.892362,0.0,0,0.0,0,0.0,0
1,1,session_2026-07-07,ALM,2026-07-07 14:25:00+00:00,2026-07-07 14:26:00+00:00,14.910,14.850,14.910,14.874651,14.880363,0.0,0,0.0,0,0.0,0
2,2,session_2026-07-07,ALM,2026-07-07 14:26:00+00:00,2026-07-07 14:27:00+00:00,14.855,14.855,14.965,14.918206,14.908449,0.0,0,0.0,0,0.0,0
3,3,session_2026-07-07,ALM,2026-07-07 14:27:00+00:00,2026-07-07 14:28:00+00:00,14.960,14.940,14.980,14.964288,14.959155,0.0,0,0.0,0,0.0,0
4,4,session_2026-07-07,ALM,2026-07-07 14:28:00+00:00,2026-07-07 14:29:00+00:00,14.985,14.980,15.045,15.024051,15.011449,0.0,0,0.0,0,0.0,0


## 3. 민감도 Artifact 저장

In [5]:
paths = {
    'metadata': OUTPUT_ROOT / f'{VERSION}_metadata.parquet',
    'comparison': OUTPUT_ROOT / f'{VERSION}_comparison.parquet',
    'schema': OUTPUT_ROOT / f'{VERSION}_schema.json',
}
variant.to_parquet(paths['metadata'], index=False, compression='zstd')
comparison.reset_index().to_parquet(paths['comparison'], index=False, compression='zstd')
schema = {
    'version': VERSION,
    'base_version': BASE_VERSION,
    'entry_distribution': 'continuous uniform between t+1 low and high',
    'random_seed': RANDOM_SEED,
    'random_draws_per_sample': RANDOM_DRAWS,
    'take_profit_return': TAKE_PROFIT_RETURN,
    'horizons_minutes': HORIZONS,
    'model_input_changed': False,
    'label_only_ablation': True,
    'intrabar_order_known': False,
    'warning': 't+1 high can occur before simulated entry; do not treat as executable ground truth',
}
paths['schema'].write_text(json.dumps(schema, ensure_ascii=False, indent=2), encoding='utf-8')
display(pd.DataFrame([{'artifact': key, 'path': str(path), 'size_mb': path.stat().st_size / 1024**2} for key, path in paths.items()]))

,artifact,path,size_mb
0,metadata,/home/user/urbandatalab/YSLee/results/preproce...,1.634197
1,comparison,/home/user/urbandatalab/YSLee/results/preproce...,0.005937
2,schema,/home/user/urbandatalab/YSLee/results/preproce...,0.000466


## 결론 및 이후 학습 비교

입력 sequence는 바뀌지 않고 target만 달라진다. 이후 동일한 날짜 split과 동일 모델로 다음 두 실험을 비교한다.

1. Baseline: `t+1 open` 기준 hard +3% label
2. Ablation: low~high 균등 진입의 `p_tp3_random_entry` soft label

랜덤 진입 label이 좋아 보이더라도 intrabar 순서를 알 수 없으므로 baseline을 대체하지 않는다. 실제 tick 또는 quote 데이터 없이 체결 현실성이 개선됐다고 결론내리지 않는다.